# Trash Sort — YOLOv11n Training
Run cells top to bottom. If the session crashes, skip to **Cell 5** to resume.

In [ ]:
# ── Cell 1: Mount Drive & install dependencies ──────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics wandb

In [ ]:
# ── Cell 2: Copy dataset from Drive to local storage (faster training) ───────
import shutil
from pathlib import Path

DRIVE_DATASET = Path('/content/drive/MyDrive/TrashDataset')
LOCAL_DATASET = Path('/content/TrashDataset')

if LOCAL_DATASET.exists():
    shutil.rmtree(LOCAL_DATASET)

print('Copying dataset from Drive to local storage...')
shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)
print('Done.')

In [ ]:
# ── Cell 3: Prepare dataset (split + generate data.yaml) ────────────────────
import random
import shutil
from pathlib import Path

DATASET_DIR = Path('/content/TrashDataset')
IMAGES_DIR  = DATASET_DIR / 'images'
LABELS_DIR  = DATASET_DIR / 'labels'
SPLIT_DIR   = DATASET_DIR / 'split'

RANDOM_SEED = 42
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
CLASS_NAMES = ['carton', 'tin', 'can']

def collect_pairs():
    images = sorted(IMAGES_DIR.glob('*.jpg')) + sorted(IMAGES_DIR.glob('*.png'))
    paired = [img for img in images if (LABELS_DIR / img.with_suffix('.txt').name).exists()]
    print(f'Found {len(paired)} paired samples.')
    return paired

def split(pairs):
    rng = random.Random(RANDOM_SEED)
    shuffled = pairs[:]
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    return shuffled[:n_train], shuffled[n_train:n_train + n_val], shuffled[n_train + n_val:]

def copy_split(pairs, subset):
    img_dst = SPLIT_DIR / subset / 'images'
    lbl_dst = SPLIT_DIR / subset / 'labels'
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)
    for img_path in pairs:
        shutil.copy2(img_path, img_dst / img_path.name)
        lbl_src = LABELS_DIR / img_path.with_suffix('.txt').name
        shutil.copy2(lbl_src, lbl_dst / lbl_src.name)

if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

pairs = collect_pairs()
train, val, test = split(pairs)

for subset, bucket in [('train', train), ('val', val), ('test', test)]:
    copy_split(bucket, subset)
    print(f'  {subset:5s}: {len(bucket)} samples')

yaml_path = SPLIT_DIR / 'data.yaml'
yaml_path.write_text('\n'.join([
    f'path: {SPLIT_DIR}',
    'train: train/images',
    'val:   val/images',
    'test:  test/images',
    '',
    f'nc: {len(CLASS_NAMES)}',
    f'names: {CLASS_NAMES}',
    '',
]))
print(f'Wrote {yaml_path}')

In [ ]:
# ── Cell 4: Login to wandb ───────────────────────────────────────────────────
import wandb
wandb.login()  # paste your API key when prompted

In [ ]:
# ── Cell 5: Train (resume-safe) ──────────────────────────────────────────────
# If session crashed: run Cell 1 + this cell only (skip 2 & 3 if Drive is still mounted).
# Set RESUME = True to continue from last checkpoint.
from pathlib import Path
from ultralytics import YOLO
import wandb

DRIVE_RUNS  = Path('/content/drive/MyDrive/trash-sort-runs')  # weights saved here
DATA_YAML   = Path('/content/TrashDataset/split/data.yaml')
RUN_NAME    = 'trash_v1'
BASE_MODEL  = 'yolo11n.pt'
LAST_PT     = DRIVE_RUNS / RUN_NAME / 'weights' / 'last.pt'

RESUME = LAST_PT.exists()  # auto-detect: resume if checkpoint exists
print(f'Resume: {RESUME} (checkpoint: {LAST_PT})')

EPOCHS   = 100
BATCH    = 16
IMGSZ    = 640
LR0      = 0.01
PATIENCE = 20
DEVICE   = 0  # GPU

wandb.init(project='trash-sort', name=RUN_NAME, resume='allow', config={
    'model':    BASE_MODEL,
    'epochs':   EPOCHS,
    'batch':    BATCH,
    'imgsz':    IMGSZ,
    'lr0':      LR0,
    'patience': PATIENCE,
    'device':   DEVICE,
})

model = YOLO(str(LAST_PT) if RESUME else BASE_MODEL)

model.train(
    data      = str(DATA_YAML),
    epochs    = EPOCHS,
    batch     = BATCH,
    imgsz     = IMGSZ,
    lr0       = LR0,
    patience  = PATIENCE,
    device    = DEVICE,
    project   = str(DRIVE_RUNS),
    name      = RUN_NAME,
    exist_ok  = True,
    resume    = RESUME,
)

wandb.finish()
print(f'\nBest weights: {DRIVE_RUNS / RUN_NAME / "weights" / "best.pt"}')